*© Sabrine Bendimerad · [sbendimerad.github.io](https://sbendimerad.github.io) · Licence [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/): you may share and adapt this material with credit, for non-commercial use, under the same licence. Datasets keep their own licences.*

# Topic modeling with LSA

A topic model finds the themes of a collection of texts without being told what they are. **LSA** (*Latent Semantic Analysis*, Landauer and Dumais, 1997) does it with one tool of linear algebra, the singular value decomposition (SVD): the same tool as PCA, applied to a table of words.

In [ ]:
import time

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.decomposition import NMF, TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# colour palette of the course
pio.templates["course"] = go.layout.Template(
    layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"]
)
pio.templates.default = "plotly_white+course"

## What is a topic?

A *topic model* is a statistical model that discovers the abstract "topics" of a collection of documents ([Wikipedia](https://en.wikipedia.org/wiki/Topic_model)). It is unsupervised: nobody tells it which topics exist. Some uses:

* **News:** tag articles automatically as *politics*, *sport*, *science*...
* **Customer support:** find what a message is about, to send it to the right team.
* **Exploration:** get a quick idea of what thousands of reviews, survey answers or emails talk about.

A few words we will use: the **corpus** is the collection of texts, a **document** is one text of the corpus, a **term** is a word of the vocabulary.

:::tip Intuition
A topic is a group of words that often appear **together**: *rocket, orbit, astronaut* on one side, *team, goal, game* on the other. A document is a mix of topics: a news story about a hockey team visiting a space center uses both groups. LSA finds these groups only by looking at which words appear in the same documents. It knows no dictionary and no labels.
:::

## The document-term matrix

A model needs numbers. As in the NLP section, each document becomes a row of numbers, with one column per term of the vocabulary: its count, or its TF-IDF. Stacked together, these rows form the **document-term matrix** $A$: $n$ documents × $m$ terms.

#### See it with data 🔬: a tiny corpus with two themes

8 short documents: 4 about space, 3 about hockey, and a last one that mixes both. We use plain counts (`CountVectorizer`) so that the matrix is easy to read, and we drop the English stop words.

In [ ]:
docs = [
    "The rocket launch put a satellite in orbit.",
    "Astronauts on the space station watched the rocket launch.",
    "The space station stays in orbit around the Earth.",
    "A new rocket will carry astronauts to the space station.",
    "The team won the hockey game in overtime.",
    "The goalie stopped every shot and the team won the game.",
    "Hockey fans cheered when the team scored a goal.",
    "After the hockey game, the team watched a rocket launch.",
]
names = [f"doc {i}" for i in range(len(docs))]

vectorizer = CountVectorizer(stop_words="english")
A_sparse = vectorizer.fit_transform(docs)
words = vectorizer.get_feature_names_out()
A = A_sparse.toarray()

print("Document-term matrix:", A.shape[0], "documents x", A.shape[1], "terms")
print(f"Share of zeros: {(A == 0).mean():.0%}")
fig = px.imshow(pd.DataFrame(A, index=names, columns=words), text_auto=True,
                color_continuous_scale=["#FFFFFF", "#3B47C4"], aspect="auto")
fig.update_layout(title="The document-term matrix A (word counts)", coloraxis_showscale=False, height=380)
fig.show()

Document-term matrix: 8 documents x 24 terms
Share of zeros: 77%


**What this shows:** 8 short documents already need 24 columns, and 77% of the cells are zeros: each document uses only a few words of the vocabulary. The space words (*rocket, orbit, station*) fill the top rows and the hockey words (*team, game, hockey*) the bottom rows. `doc 7` uses both groups.

### Sparse matrices

On a real corpus the vocabulary has thousands of words and a document uses a few dozen: the matrix is almost only zeros. Such a matrix is called **sparse**. Storing all those zeros wastes memory, and multiplying by them wastes time.

So `CountVectorizer` and `TfidfVectorizer` return a **scipy sparse matrix**: it stores only the non-zero values, with their positions (here in the *CSR* format: the values, their column indices, and where each row starts).

#### See it with data 🔬: dense vs sparse on 2,400 real messages

The corpus of this section: 2,400 messages from 6 discussion groups of the *20 Newsgroups* dataset (you will use it in the next lesson). We build its TF-IDF matrix, then compare the memory and the time of a matrix product ($A A^\top$, the similarity of every pair of documents) in the sparse and dense formats.

In [ ]:
news = pd.read_csv("data/newsgroups.csv")
X = TfidfVectorizer(stop_words="english", min_df=2).fit_transform(news["text"])
n_cells = X.shape[0] * X.shape[1]
print(f"TF-IDF matrix: {X.shape[0]:,} documents x {X.shape[1]:,} terms = {n_cells:,} cells")
print(f"Non-zero values: {X.nnz:,}, so {1 - X.nnz / n_cells:.2%} of zeros")

sparse_mb = (X.data.nbytes + X.indices.nbytes + X.indptr.nbytes) / 1e6
dense_mb = n_cells * 8 / 1e6                         # 8 bytes per float64 number
print(f"Memory: sparse {sparse_mb:.1f} MB, dense {dense_mb:.1f} MB ({dense_mb / sparse_mb:.0f} times more)")

X_dense = X.toarray()
start = time.perf_counter()
X @ X.T
sparse_time = time.perf_counter() - start
start = time.perf_counter()
X_dense @ X_dense.T
dense_time = time.perf_counter() - start
print(f"A @ A.T is {dense_time / sparse_time:.0f} times faster with the sparse matrix (on this computer)")
del X_dense                                          # free the memory

TF-IDF matrix: 2,400 documents x 10,424 terms = 25,017,600 cells
Non-zero values: 105,427, so 99.58% of zeros
Memory: sparse 1.3 MB, dense 200.1 MB (157 times more)


A @ A.T is 4 times faster with the sparse matrix (on this computer)


**What this shows:** the matrix has 2,400 × 10,424 = 25,017,600 cells, but only 105,427 are not zero: 99.58% of zeros. The sparse format needs 1.3 MB instead of 200.1 MB, 157 times less. The product is also several times faster in the sparse format (the exact ratio depends on your computer). Keep your text matrices sparse: call `.toarray()` only on small matrices, to look at them.

## SVD and truncated SVD

### The decomposition

Any matrix $A$ ($n$ documents × $m$ terms) can be written as a product of three matrices, its **singular value decomposition**:

$$A = U\,\Sigma\,V^\top$$

* $U$ ($n \times r$): **documents × topics**. Row $i$ gives the coordinates of document $i$ on each topic.
* $\Sigma$ ($r \times r$): a diagonal matrix of positive numbers $\sigma_1 \ge \sigma_2 \ge \dots \ge \sigma_r$, the **singular values**: the strength of each topic.
* $V^\top$ ($r \times m$): **topics × terms**. Row $j$ gives the weight of each word in topic $j$.

$r$ is the rank of $A$ (at most the smaller of $n$ and $m$). The columns of $U$ are orthogonal to each other and of length 1 ($U^\top U = I$), and so are the columns of $V$ ($V^\top V = I$).

**The link with PCA.** With these properties,

$$A^\top A = V\Sigma U^\top U \Sigma V^\top = V\,\Sigma^2\,V^\top \qquad\qquad A A^\top = U\,\Sigma^2\,U^\top$$

$A^\top A$ ($m \times m$) says which words appear in the same documents, $AA^\top$ ($n \times n$) which documents share words. So $V$ holds the eigenvectors of the word-word matrix and $U$ those of the document-document matrix, exactly like the principal components of PCA. The one difference: PCA first centers each column (subtracts its mean), LSA does not. Centering would replace almost all the zeros by non-zero values: the matrix would no longer be sparse.

### Truncated SVD

The topics come sorted by strength. **Truncated SVD** keeps only the $k$ strongest ones ($k < r$), and drops the rest:

$$A \approx A_k = U_k\,\Sigma_k\,V_k^\top$$

![Diagram of truncated SVD: the document-term matrix A is approximately U (documents by topics) times Sigma (the topic strengths, a diagonal of decreasing values) times V transposed (topics by words); only the first k columns of U, the first k values of Sigma and the first k rows of V transposed are kept, the rest is dropped]()

Among all matrices built from only $k$ topics, $A_k$ is the closest to $A$ (the Eckart–Young theorem). In scikit-learn, `TruncatedSVD(n_components=k)` computes it directly on a sparse matrix:

* `fit_transform(A)` returns $U_k\Sigma_k$: the coordinates of each document on the $k$ topics;
* `components_` is $V_k^\top$: the weight of each word in each topic.

:::tip Intuition
Think of the 24 columns of our tiny matrix as 24 questions asked to each document: "do you say *rocket*? *team*? *orbit*?..." Many questions are redundant: a document that says *rocket* often says *launch* too. SVD replaces the 24 questions by a few new ones, each a weighted mix of words ("how much do you sound like space?"), chosen to keep as much of the table as possible. These new questions are the topics.
:::

#### See it with data 🔬: SVD by hand, then with scikit-learn

We decompose the tiny matrix with NumPy, check that the product gives $A$ back, then compare with `TruncatedSVD` and 2 topics.

In [ ]:
U, sigma, Vt = np.linalg.svd(A, full_matrices=False)
print("Shapes: U", U.shape, " Sigma", sigma.shape, " Vt", Vt.shape)
print("Singular values:", sigma.round(2))
print("U Sigma Vt gives A back:", np.allclose(U @ np.diag(sigma) @ Vt, A))
print("Columns of U orthogonal and of length 1:", np.allclose(U.T @ U, np.eye(len(sigma))))

svd = TruncatedSVD(n_components=2, random_state=0)
doc_topics = svd.fit_transform(A_sparse)             # U_k Sigma_k: documents x topics
topic_words = svd.components_                        # Vt_k: topics x words
print("\nscikit-learn, singular values:", svd.singular_values_.round(2))
print("Same document coordinates as U_k Sigma_k (up to the sign):",
      np.allclose(np.abs(doc_topics), np.abs(U[:, :2] * sigma[:2])))
print("Same word weights as the first 2 rows of Vt (up to the sign):", np.allclose(np.abs(topic_words), np.abs(Vt[:2])))

Shapes: U (8, 8)  Sigma (8,)  Vt (8, 24)
Singular values: [3.77 3.31 2.26 2.2  1.93 1.56 1.38 0.9 ]
U Sigma Vt gives A back: True
Columns of U orthogonal and of length 1: True

scikit-learn, singular values: [3.77 3.31]
Same document coordinates as U_k Sigma_k (up to the sign): True
Same word weights as the first 2 rows of Vt (up to the sign): True


**What this shows:** the 8 × 24 matrix has 8 singular values, from 3.77 down to 0.9, and $U\Sigma V^\top$ gives $A$ back exactly. `TruncatedSVD` with 2 components finds the same two strongest values (3.77 and 3.31), the same document coordinates and the same word weights, *up to the sign*: more on that in the limits below.

#### See it with data 🔬: the words of each topic

Each row of `components_` gives a weight to each of the 24 words. Let's draw them.

In [ ]:
weights = pd.DataFrame(topic_words.T, index=words, columns=["topic 1", "topic 2"])
fig = make_subplots(rows=1, cols=2, subplot_titles=["Topic 1", "Topic 2"], horizontal_spacing=0.15)
for j, col in enumerate(weights.columns):
    w = weights[col].sort_values()
    fig.add_trace(go.Bar(x=w.values, y=w.index, orientation="h", showlegend=False,
                         marker_color=np.where(w.values > 0, "#3B47C4", "#E4572E")), row=1, col=j + 1)
fig.update_xaxes(title_text="weight of the word in the topic")
fig.update_layout(title="components_: the weight of each word in each topic", height=600)
fig.show()
for col in weights.columns:
    w = weights[col].sort_values()
    print(f"{col}: largest weights", ", ".join(f"{word} {v:.2f}" for word, v in w[::-1][:3].items()),
          "| smallest", ", ".join(f"{word} {v:.2f}" for word, v in w[:3].items()))

topic 1: largest weights rocket 0.44, team 0.35, launch 0.34 | smallest stays 0.06, earth 0.06, goal 0.06
topic 2: largest weights team 0.44, game 0.33, hockey 0.31 | smallest station -0.33, space -0.33, astronauts -0.24


**What this shows:**

* **Topic 1** gives a positive weight to every word. The largest go to the most frequent words, from both themes: *rocket* (0.44), *team* (0.35), *launch* (0.34). It is not a theme: it is the "general direction" of the corpus, roughly "how many of the usual words does this document use".
* **Topic 2** is the interesting one: it is a **contrast**. Hockey words get positive weights (*team* 0.44, *game* 0.33, *hockey* 0.31), space words negative ones (*space* and *station* −0.33, *astronauts* −0.24). The sign tells the theme.

#### See it with data 🔬: the documents in the topic space

`fit_transform` gave each document two coordinates. Let's plot them.

In [ ]:
coords = pd.DataFrame(doc_topics, columns=["topic 1", "topic 2"], index=names)
coords["theme"] = ["space"] * 4 + ["hockey"] * 3 + ["both"]
fig = px.scatter(coords, x="topic 1", y="topic 2", color="theme", text=coords.index, hover_data={"theme": True})
fig.update_traces(marker_size=12, textposition="top center")
fig.add_hline(y=0, line_color="#999999", line_dash="dash")
fig.update_layout(title="The 8 documents in the space of the 2 topics", height=450)
fig.show()
coords.round(2)

,topic 1,topic 2,theme
doc 0,0.97,-0.50,space
doc 1,1.86,-1.27,space
doc 2,0.82,-0.97,space
doc 3,1.46,-1.37,space
doc 4,1.16,1.49,hockey
doc 5,1.01,1.44,hockey
doc 6,0.88,1.19,hockey
doc 7,1.97,0.72,both


**What this shows:** on topic 2, the four space documents are negative (−0.50 to −1.37) and the three hockey documents positive (1.19 to 1.49). `doc 7` (hockey *and* a rocket launch) sits in between, at 0.72: LSA sees it as a mix. On topic 1, the documents with the most words of the vocabulary go furthest to the right: `doc 7` (1.97) and `doc 1` (1.86). In 2 numbers per document, we keep the theme.

#### See it with data 🔬: how good is the approximation?

$A_k = U_k\Sigma_kV_k^\top$ rebuilds the matrix from $k$ topics. We measure the relative error $\|A - A_k\| / \|A\|$ (the square root of the sum of the squared differences, divided by the same for $A$) for $k = 1$ to 8.

In [ ]:
rows = []
for k in range(1, len(sigma) + 1):
    A_k = U[:, :k] @ np.diag(sigma[:k]) @ Vt[:k]
    rows.append({"k": k, "relative error": np.linalg.norm(A - A_k) / np.linalg.norm(A),
                 "share of the sum of squares kept": (sigma[:k] ** 2).sum() / (sigma ** 2).sum()})
errors = pd.DataFrame(rows)

fig = px.line(errors, x="k", y=["relative error", "share of the sum of squares kept"], markers=True)
fig.update_layout(title="Rebuilding A from its k strongest topics", xaxis_title="number of topics k",
                  yaxis_title="", legend_title="", height=380)
fig.show()
errors.round(3).set_index("k").T

k,1,2,3,4,5,6,7,8
relative error,0.822,0.654,0.558,0.449,0.343,0.249,0.136,0.0
share of the sum of squares kept,0.324,0.572,0.689,0.798,0.883,0.938,0.981,1.0


In [ ]:
A_2 = U[:, :2] @ np.diag(sigma[:2]) @ Vt[:2]
fig = make_subplots(rows=1, cols=2, subplot_titles=["A (the real counts)", "A_2 (rebuilt from 2 topics)"],
                    horizontal_spacing=0.08)
for j, M in enumerate([A, A_2]):
    fig.add_trace(go.Heatmap(z=M, x=words, y=names, zmin=-0.5, zmax=1.5, coloraxis="coloraxis"), row=1, col=j + 1)
fig.update_yaxes(autorange="reversed")
fig.update_layout(coloraxis=dict(colorscale=[[0, "#E4572E"], [0.25, "#FFFFFF"], [1, "#3B47C4"]], cmin=-0.5, cmax=1.5),
                  height=420,
                  title="The real matrix and its approximation with 2 topics")
fig.show()

**What this shows:** with 1 topic the relative error is 0.822, with 2 topics 0.654, and it reaches 0 with all 8 (then $A_k = A$). Each topic adds less than the previous one, since the singular values decrease. On the heatmaps, $A_2$ is a blurred version of $A$: it no longer knows that `doc 0` says *satellite*, but it knows that `doc 0` belongs to the space block and gives some weight to all the space words, even those the document does not use. That's the point of LSA: documents that use *different* words of the same theme become similar. Notice also the small negative values (orange) in $A_2$: a count can't be negative, a sign that topics are not easy to read.

## The limits of LSA

* **The sign of a topic is arbitrary.** Multiplying a column of $U$ and the same row of $V^\top$ by −1 gives exactly the same product. So "space is negative on topic 2" means nothing in itself: another run, or another library, can flip it. Only the contrast (space on one side, hockey on the other) matters.
* **The first topic is often a "general size" topic.** Since the data are not centered, the first component points towards the average document. Its coordinate mostly says how much a document uses the frequent words of the corpus, whatever its theme.
* **Topics mix words with positive and negative weights.** A topic is a contrast between groups of words, not a list of words. With many topics, several themes get mixed in each component, and naming them is hard.
* **Bag of words.** The order of the words is lost, and a word with two meanings (*bank*, *game*...) gets one single position.

#### See it with data 🔬: a flipped sign and the "size" topic

In [ ]:
U_flip, Vt_flip = U.copy(), Vt.copy()
U_flip[:, 1] *= -1                                   # flip topic 2 in U...
Vt_flip[1] *= -1                                     # ...and in Vt
print("Same product after flipping the sign of topic 2:", np.allclose(U_flip @ np.diag(sigma) @ Vt_flip, A))

average_doc = A.mean(axis=0)                         # the average row of A
print("Cosine between topic 1 (first row of Vt) and the average document:",
      round(topic_words[0] @ average_doc / np.linalg.norm(average_doc), 2))
print("Correlation between the topic 1 coordinate and the similarity to the average document (A @ average):",
      round(np.corrcoef(doc_topics[:, 0], A @ average_doc)[0, 1], 2))
print("explained_variance_ratio_ of the 2 topics:", svd.explained_variance_ratio_.round(3))

Same product after flipping the sign of topic 2: True
Cosine between topic 1 (first row of Vt) and the average document: 0.98
Correlation between the topic 1 coordinate and the similarity to the average document (A @ average): 0.98
explained_variance_ratio_ of the 2 topics: [0.046 0.354]


**What this shows:** flipping the sign of topic 2 in both $U$ and $V^\top$ gives exactly the same matrix. Topic 1 points almost exactly towards the average document (cosine 0.98), and the topic 1 coordinate of a document is almost the same thing as its similarity to that average document (correlation 0.98). It measures *how much* a document uses the common words of the corpus, not *what* it talks about. That is also why its `explained_variance_ratio_` is tiny (0.046, against 0.354 for topic 2): scikit-learn computes this ratio on the variance of the coordinates around their mean, and topic 1 mostly captures the mean itself. Don't be surprised if the ratios are not sorted.

## Alternatives: NMF and LDA

**NMF** (*Non-negative Matrix Factorization*) also writes $A \approx W H$ (documents × topics, topics × words), but forces all the numbers to be **positive or zero**. No contrasts, no arbitrary signs: a document is a sum of topics, and a topic a list of words. It is often easier to read. **LDA** (*Latent Dirichlet Allocation*) is a probabilistic model: each document is a mixture of topics, each topic a probability distribution over words (`LatentDirichletAllocation` in scikit-learn, on word counts).

#### See it with data 🔬: NMF on the tiny corpus

In [ ]:
nmf = NMF(n_components=2, init="nndsvda", max_iter=1000, random_state=0)
doc_topics_nmf = nmf.fit_transform(A_sparse)         # W: documents x topics
for j, row in enumerate(nmf.components_):            # H: topics x words
    print(f"NMF topic {j + 1}:", ", ".join(words[np.argsort(row)[::-1][:5]]))
pd.DataFrame(doc_topics_nmf, index=names, columns=["NMF topic 1", "NMF topic 2"]).round(2)

NMF topic 1: rocket, space, station, astronauts, launch
NMF topic 2: team, game, hockey, won, overtime


,NMF topic 1,NMF topic 2
doc 0,0.41,0.04
doc 1,0.86,0.03
doc 2,0.48,0.00
doc 3,0.77,0.00
doc 4,0.00,0.99
doc 5,0.00,0.92
doc 6,0.00,0.77
doc 7,0.34,0.85


**What this shows:** NMF finds one space topic (*rocket, space, station, astronauts, launch*) and one hockey topic (*team, game, hockey, won, overtime*), with no "size" topic and no negative weights. The four space documents get 0 or almost 0 on the hockey topic, the three hockey documents exactly 0 on the space topic, and `doc 7` gets both (0.34 and 0.85). LSA remains the simplest and fastest method, and the one that builds directly on PCA; NMF is worth a try when you need topics that you can name.

## Summary

* LSA = a truncated SVD of the document-term matrix (counts or TF-IDF): PCA applied to word importances, without centering, so the matrix stays sparse.
* $A \approx U_k \Sigma_k V_k^\top$: `fit_transform` gives the documents × topics coordinates ($U_k\Sigma_k$), `components_` the topics × words weights ($V_k^\top$).
* A topic is a combination of words. The first topics keep the most of the matrix, the next ones less and less.
* Topics are not directly interpretable: arbitrary signs, a first "size" topic, contrasts between groups of words. Read their top words carefully, and to group documents, run a clustering (k-means...) on their topic coordinates.